# 07 · Modelo final HPO y validación externa

Este notebook evalúa las configuraciones congeladas en el notebook 06b sobre **octubre de 2022**, un periodo posterior que no intervino en el HPO ni en el refinamiento. Se comparan XGBoost refinado y una regresión logística L2 como referencia.

El objetivo es medir generalización temporal y cerrar la elección del modelo antes del test final. **Noviembre y diciembre de 2022 no se leen en este notebook**: quedan reservados para la única evaluación definitiva del notebook 08. Los datos de 2023 se limitan al análisis descriptivo de viajes porque no existen estados de estación con los que construir las etiquetas de riesgo.

## Resultado que llega del notebook 06b

- XGBoost refinado: F1 macro medio 0,8130 y balanced accuracy media 0,8305.
- Regresión logística refinada: F1 macro medio 0,7727 y balanced accuracy media 0,7342.
- La mejora de XGBoost respecto al HPO original es pequeña; el refinamiento confirma una región estable, no un salto de rendimiento.
- La variación entre folds es mayor que la variación entre semillas. Por ello, octubre se usa como comprobación temporal externa.
- El trial elegido prioriza balanced accuracy entre configuraciones situadas a menos de 0,003 del mejor F1 macro. Sus iteraciones óptimas internas fueron 254, 582, 325 y 362; se congela su media redondeada, 381 árboles, para no usar octubre como conjunto de early stopping.

## Entorno, rutas y configuración

La muestra de train mantiene el coste del ajuste final bajo control y se obtiene antes de cualquier transformación. Cambiar `MAX_TRAIN_ROWS` altera el experimento y debe quedar documentado.

In [ ]:
# %pip install scikit-learn xgboost matplotlib

from pathlib import Path
import gc
import json
import time
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, log_loss,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
data_candidates = [
    PROJECT_ROOT / 'notebooks' / 'Datos modelado',
    PROJECT_ROOT / 'Datos modelado',
]
MODEL_DATA_DIR = next(
    (path for path in data_candidates
     if (path / 'hpo_refinamiento' / 'mejores_hiperparametros_refinados.csv').exists()),
    None,
)
if MODEL_DATA_DIR is None:
    raise FileNotFoundError('No se encontraron los resultados completos del notebook 06b.')

FEATURES_DIR = MODEL_DATA_DIR / 'estacion_hora_features'
HPO_DIR = MODEL_DATA_DIR / 'hpo_refinamiento'
OUTPUT_DIR = MODEL_DATA_DIR / 'validacion_externa'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'risk_class_1h'
TIME_COLUMN = 'fecha_hora_local'
RANDOM_STATE = 42
CHUNK_SIZE = 100_000
MAX_TRAIN_ROWS = 500_000
VALIDATION_PERIOD = '202210'
CLASS_NAMES = {0: 'estable', 1: 'riesgo_vaciado', 2: 'riesgo_saturacion'}

print('Carpeta de datos:', MODEL_DATA_DIR)
print('Resultados:', OUTPUT_DIR)
print('Versiones:', {'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__})

## Variables y barreras temporales

Las variables futuras y las que definen directamente la etiqueta quedan fuera. Los archivos de entrenamiento terminan en septiembre de 2022. La validación externa se limita físicamente a octubre y exige `dataset_split == 'validation'`.

In [ ]:
NUMERIC_FEATURES = [
    'capacity', 'bikes_available', 'docks_available', 'reservations_count',
    'occupancy_ratio', 'light', 'weather_available',
    'uv_radiation_median_mw_m2', 'wind_speed_median_m_s',
    'wind_direction_sin_mean', 'wind_direction_cos_mean',
    'temperature_median_c', 'relative_humidity_median_pct',
    'barometric_pressure_median_mb', 'solar_radiation_median_w_m2',
    'precipitation_mean_l_m2', 'precipitation_max_l_m2',
    'n_temperature', 'n_relative_humidity', 'n_precipitation',
    'hour', 'day_of_week', 'month', 'week_of_year',
    'occupancy_ratio_lag_1h', 'occupancy_ratio_lag_2h', 'occupancy_ratio_lag_24h',
    'bikes_available_lag_1h', 'bikes_available_lag_2h', 'bikes_available_lag_24h',
    'net_flow_lag_1h', 'net_flow_lag_2h', 'net_flow_lag_24h',
    'departures_count_lag_1h', 'departures_count_lag_2h', 'departures_count_lag_24h',
    'arrivals_count_lag_1h', 'arrivals_count_lag_2h', 'arrivals_count_lag_24h',
    'occupancy_ratio_mean_previous_3h', 'occupancy_ratio_mean_previous_24h',
    'net_flow_mean_previous_3h', 'net_flow_mean_previous_24h',
    'departures_count_mean_previous_3h', 'departures_count_mean_previous_24h',
    'arrivals_count_mean_previous_3h', 'arrivals_count_mean_previous_24h',
]
CATEGORICAL_FEATURES = ['station_id', 'tipo_dia']
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
METADATA_COLUMNS = [TIME_COLUMN, 'station_id']
READ_COLUMNS = list(dict.fromkeys(MODEL_FEATURES + METADATA_COLUMNS + [TARGET, 'dataset_split']))

def period_from_file(file_path: Path) -> str:
    return file_path.stem.rsplit('_', maxsplit=1)[-1]

all_feature_files = sorted(FEATURES_DIR.glob('estacion_hora_features_*.csv'))
train_files = [
    path for path in all_feature_files
    if period_from_file(path)[:4] in {'2019', '2021'}
    or ('202201' <= period_from_file(path) <= '202209')
]
validation_files = [path for path in all_feature_files if period_from_file(path) == VALIDATION_PERIOD]

if not train_files or len(validation_files) != 1:
    raise FileNotFoundError('Faltan las particiones de train o la partición única de octubre de 2022.')
assert period_from_file(train_files[-1]) == '202209'
assert all(period_from_file(path) < VALIDATION_PERIOD for path in train_files)
assert all(not period_from_file(path).startswith('2020') for path in train_files)
assert period_from_file(validation_files[0]) == '202210'
assert not any(period_from_file(path) in {'202211', '202212'} for path in train_files + validation_files)

print(f'Particiones de train: {len(train_files)}; última: {train_files[-1].name}')
print('Validación externa:', validation_files[0].name)

## Configuraciones congeladas

Los parámetros se leen de los artefactos del 06b. Para XGBoost, `n_estimators=600` fue el techo de búsqueda; el número efectivo se fija en la media de las iteraciones óptimas internas del trial elegido. Octubre no actúa como `eval_set` y no decide cuándo detener el entrenamiento.

In [ ]:
parameter_table = pd.read_csv(
    HPO_DIR / 'mejores_hiperparametros_refinados.csv', encoding='utf-8-sig'
)
selection_table = pd.read_csv(
    HPO_DIR / 'configuraciones_refinadas_seleccionadas.csv', encoding='utf-8-sig'
)
audit_06b = pd.read_csv(HPO_DIR / 'auditoria_refinamiento_sin_fuga.csv', encoding='utf-8-sig')

def parameters_for(model_name: str) -> dict:
    rows = parameter_table.loc[parameter_table['model'].eq(model_name)]
    return dict(zip(rows['parameter'], rows['value']))

xgb_parameters = parameters_for('xgboost_refined')
logistic_parameters = parameters_for('logistic_refined')
for integer_parameter in ('n_estimators', 'max_depth'):
    xgb_parameters[integer_parameter] = int(xgb_parameters[integer_parameter])

selected_xgb_row = selection_table.loc[selection_table['model'].eq('xgboost_refined')].iloc[0]
FROZEN_N_ESTIMATORS = int(round(selected_xgb_row['best_iteration_mean']))
assert FROZEN_N_ESTIMATORS == 381
assert str(audit_06b.loc[audit_06b['control'].eq('test_consultado'), 'value'].iloc[0]).lower() == 'false'
assert str(audit_06b.loc[audit_06b['control'].eq('ultimo_mes_leido'), 'value'].iloc[0]) == '202209'

frozen_configuration = {
    'xgboost_refined': {**xgb_parameters, 'n_estimators': FROZEN_N_ESTIMATORS},
    'logistic_refined': {
        'C': float(logistic_parameters['C']), 'solver': 'lbfgs',
        'max_iter': 700, 'tol': 1e-3, 'class_weight': None,
    },
}
print(json.dumps(frozen_configuration, indent=2, ensure_ascii=False))

## Carga reproducible de train y validación externa

Se realiza una primera pasada para contar las filas elegibles y una segunda para seleccionar una muestra aleatoria reproducible que conserva aproximadamente la distribución de clases. La validación externa se carga completa.

In [ ]:
train_class_counts = {0: 0, 1: 0, 2: 0}
for file_path in train_files:
    for chunk in pd.read_csv(
        file_path, usecols=[TARGET, 'dataset_split'], chunksize=CHUNK_SIZE, low_memory=False,
    ):
        eligible = chunk.loc[chunk['dataset_split'].eq('train') & chunk[TARGET].notna(), TARGET].astype(int)
        for class_value, count in eligible.value_counts().items():
            train_class_counts[int(class_value)] += int(count)

total_train_rows = sum(train_class_counts.values())
sampling_probability = min(1.0, MAX_TRAIN_ROWS / total_train_rows)
rng_by_class = {
    class_value: np.random.default_rng(RANDOM_STATE + class_value)
    for class_value in CLASS_NAMES
}
train_parts = []
for file_path in train_files:
    for chunk in pd.read_csv(file_path, usecols=READ_COLUMNS, chunksize=CHUNK_SIZE, low_memory=False):
        chunk = chunk.loc[chunk['dataset_split'].eq('train') & chunk[TARGET].notna()].copy()
        if chunk.empty:
            continue
        chunk[TARGET] = chunk[TARGET].astype('int8')
        selected_positions = []
        for class_value in CLASS_NAMES:
            positions = np.flatnonzero(chunk[TARGET].to_numpy() == class_value)
            if len(positions):
                selected_positions.append(
                    positions[rng_by_class[class_value].random(len(positions)) < sampling_probability]
                )
        if selected_positions:
            positions = np.concatenate(selected_positions)
            if len(positions):
                train_parts.append(chunk.iloc[positions].copy())

train = pd.concat(train_parts, ignore_index=True)
validation_parts = []
for chunk in pd.read_csv(
    validation_files[0], usecols=READ_COLUMNS, chunksize=CHUNK_SIZE, low_memory=False,
):
    chunk = chunk.loc[chunk['dataset_split'].eq('validation') & chunk[TARGET].notna()].copy()
    if not chunk.empty:
        chunk[TARGET] = chunk[TARGET].astype('int8')
        validation_parts.append(chunk)
validation = pd.concat(validation_parts, ignore_index=True)

train[TIME_COLUMN] = pd.to_datetime(train[TIME_COLUMN], errors='raise')
validation[TIME_COLUMN] = pd.to_datetime(validation[TIME_COLUMN], errors='raise')
assert train['dataset_split'].eq('train').all()
assert validation['dataset_split'].eq('validation').all()
assert train[TIME_COLUMN].max() < validation[TIME_COLUMN].min()
assert validation[TIME_COLUMN].dt.to_period('M').astype(str).eq('2022-10').all()

distribution = pd.concat([
    train[TARGET].value_counts(normalize=True).sort_index().rename('train'),
    validation[TARGET].value_counts(normalize=True).sort_index().rename('validation_externa'),
], axis=1)
print(f'Train disponible: {total_train_rows:,}; muestra cargada: {len(train):,}')
print(f'Validación externa completa: {len(validation):,}')
display(distribution)

del train_parts, validation_parts
gc.collect()

## Preprocesamiento ajustado solo con train

Cada modelo recibe un preprocesador independiente. Las medianas, indicadores de ausencia, moda y categorías one-hot se aprenden únicamente con train. La logística escala solo las variables numéricas.

In [ ]:
def make_preprocessor(scale_numeric: bool) -> ColumnTransformer:
    numeric_steps = [('imputer', SimpleImputer(strategy='median', add_indicator=True))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer(transformers=[
        ('numeric', Pipeline(numeric_steps), NUMERIC_FEATURES),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('one_hot', OneHotEncoder(handle_unknown='ignore')),
        ]), CATEGORICAL_FEATURES),
    ])

X_train_raw = train[MODEL_FEATURES]
y_train = train[TARGET].astype(int).reset_index(drop=True)
X_validation_raw = validation[MODEL_FEATURES]
y_validation = validation[TARGET].astype(int).reset_index(drop=True)

xgb_preprocessor = make_preprocessor(scale_numeric=False)
X_train_xgb = xgb_preprocessor.fit_transform(X_train_raw).astype(np.float32)
X_validation_xgb = xgb_preprocessor.transform(X_validation_raw).astype(np.float32)

logistic_preprocessor = make_preprocessor(scale_numeric=True)
X_train_logistic = logistic_preprocessor.fit_transform(X_train_raw).astype(np.float32)
X_validation_logistic = logistic_preprocessor.transform(X_validation_raw).astype(np.float32)

print('XGBoost:', X_train_xgb.shape, X_validation_xgb.shape)
print('Logística:', X_train_logistic.shape, X_validation_logistic.shape)

## Ajuste de los modelos congelados

Los pesos de las clases minoritarias pertenecen a la configuración XGBoost aprendida en train. La validación externa no se pasa a `fit` en ninguno de los dos modelos.

In [ ]:
weight_empty = float(xgb_parameters['weight_empty'])
weight_full = float(xgb_parameters['weight_full'])
xgb_model_parameters = {
    key: value for key, value in frozen_configuration['xgboost_refined'].items()
    if key not in {'weight_empty', 'weight_full'}
}
xgb_model = XGBClassifier(
    objective='multi:softprob', num_class=3, eval_metric='mlogloss',
    tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE,
    **xgb_model_parameters,
)
sample_weights = np.ones(len(y_train), dtype=np.float32)
sample_weights[y_train.to_numpy() == 1] *= weight_empty
sample_weights[y_train.to_numpy() == 2] *= weight_full

start = time.perf_counter()
xgb_model.fit(X_train_xgb, y_train, sample_weight=sample_weights)
xgb_fit_seconds = time.perf_counter() - start

logistic_model = LogisticRegression(
    C=frozen_configuration['logistic_refined']['C'], solver='lbfgs',
    class_weight=None, max_iter=700, tol=1e-3, random_state=RANDOM_STATE,
)
start = time.perf_counter()
logistic_model.fit(X_train_logistic, y_train)
logistic_fit_seconds = time.perf_counter() - start

print(f'XGBoost ajustado en {xgb_fit_seconds:.1f} s con {FROZEN_N_ESTIMATORS} árboles.')
print(f'Logística ajustada en {logistic_fit_seconds:.1f} s; iteraciones: {int(logistic_model.n_iter_.max())}.')
assert int(logistic_model.n_iter_.max()) < logistic_model.max_iter

## Evaluación externa y diagnóstico por clase

La decisión principal usa F1 macro; balanced accuracy actúa como criterio complementario. También se guardan matrices de confusión, informes por clase y probabilidades para que el resultado sea auditable.

In [ ]:
evaluation_specs = [
    ('xgboost_refined', xgb_model, X_validation_xgb, xgb_fit_seconds),
    ('logistic_refined', logistic_model, X_validation_logistic, logistic_fit_seconds),
]
metrics_rows = []
report_parts = []
confusion_parts = []
prediction_parts = []
predictions_by_model = {}

for model_name, model, features, fit_seconds in evaluation_specs:
    prediction = model.predict(features).astype(int)
    probabilities = model.predict_proba(features)
    predictions_by_model[model_name] = prediction
    metrics_rows.append({
        'model': model_name,
        'f1_macro': f1_score(y_validation, prediction, average='macro'),
        'balanced_accuracy': balanced_accuracy_score(y_validation, prediction),
        'f1_weighted': f1_score(y_validation, prediction, average='weighted'),
        'accuracy': accuracy_score(y_validation, prediction),
        'log_loss': log_loss(y_validation, probabilities, labels=[0, 1, 2]),
        'fit_seconds': fit_seconds,
        'train_rows': len(train),
        'validation_rows': len(validation),
    })

    report = pd.DataFrame(
        classification_report(
            y_validation, prediction, labels=[0, 1, 2],
            target_names=[CLASS_NAMES[i] for i in range(3)],
            output_dict=True, zero_division=0,
        )
    ).T.reset_index(names='class')
    report.insert(0, 'model', model_name)
    report_parts.append(report)

    matrix = confusion_matrix(y_validation, prediction, labels=[0, 1, 2])
    confusion_parts.append(pd.DataFrame([
        {'model': model_name, 'real': real, 'predicted': predicted, 'count': int(matrix[real, predicted])}
        for real in range(3) for predicted in range(3)
    ]))

    model_predictions = validation[METADATA_COLUMNS + [TARGET]].copy()
    model_predictions.insert(0, 'model', model_name)
    model_predictions['prediction'] = prediction
    for class_value in range(3):
        model_predictions[f'probability_{class_value}'] = probabilities[:, class_value]
    prediction_parts.append(model_predictions)

metrics = pd.DataFrame(metrics_rows).sort_values(
    ['f1_macro', 'balanced_accuracy'], ascending=False
).reset_index(drop=True)
reports = pd.concat(report_parts, ignore_index=True)
confusions = pd.concat(confusion_parts, ignore_index=True)
predictions = pd.concat(prediction_parts, ignore_index=True)
display(metrics)
display(reports.loc[reports['class'].isin(CLASS_NAMES.values())])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for axis, (model_name, prediction) in zip(axes, predictions_by_model.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_validation, prediction, labels=[0, 1, 2],
        display_labels=[CLASS_NAMES[i] for i in range(3)],
        normalize='true', values_format='.2f', cmap='Blues', ax=axis, colorbar=False,
    )
    axis.set_title(model_name.replace('_', ' ').title())
    axis.tick_params(axis='x', rotation=20)
fig.suptitle('Validación externa de octubre de 2022 · matriz normalizada por clase real')
plt.tight_layout()
plt.show()

metric_plot = metrics.set_index('model')[['f1_macro', 'balanced_accuracy']]
ax = metric_plot.plot(kind='bar', figsize=(8, 4), color=['#0B6E99', '#D97904'])
ax.set_ylim(0, 1)
ax.set_ylabel('Puntuación')
ax.set_title('Comparación en validación externa')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Selección, exportación y auditoría

El ganador se decide por F1 macro y, en caso de empate, por balanced accuracy. Esta elección cierra la selección con octubre. No autoriza a modificar hiperparámetros; cualquier cambio posterior exigiría una nueva evaluación realmente externa.

In [ ]:
selected_model = metrics.iloc[0]['model']
selection_result = metrics.iloc[[0]].copy()
selection_result.insert(0, 'selection_rule', 'f1_macro_desc_then_balanced_accuracy_desc')
selection_result['validation_period'] = VALIDATION_PERIOD

metrics.to_csv(OUTPUT_DIR / 'comparacion_validacion_externa.csv', index=False, encoding='utf-8-sig')
reports.to_csv(OUTPUT_DIR / 'informe_por_clase_validacion_externa.csv', index=False, encoding='utf-8-sig')
confusions.to_csv(OUTPUT_DIR / 'matrices_confusion_validacion_externa.csv', index=False, encoding='utf-8-sig')
predictions.to_csv(OUTPUT_DIR / 'predicciones_validacion_externa.csv', index=False, encoding='utf-8-sig')
selection_result.to_csv(OUTPUT_DIR / 'modelo_seleccionado_validacion_externa.csv', index=False, encoding='utf-8-sig')

# Se guardan artefactos de validación para reproducir el resultado; aún no son el modelo de producción.
with (OUTPUT_DIR / 'xgboost_refined_validacion_externa.pkl').open('wb') as file:
    pickle.dump({
        'model': xgb_model, 'preprocessor': xgb_preprocessor,
        'features': MODEL_FEATURES, 'classes': CLASS_NAMES,
        'frozen_configuration': frozen_configuration['xgboost_refined'],
    }, file)
with (OUTPUT_DIR / 'logistic_refined_validacion_externa.pkl').open('wb') as file:
    pickle.dump({
        'model': logistic_model, 'preprocessor': logistic_preprocessor,
        'features': MODEL_FEATURES, 'classes': CLASS_NAMES,
        'frozen_configuration': frozen_configuration['logistic_refined'],
    }, file)

audit_table = pd.DataFrame([
    {'control': 'ultimo_mes_train', 'value': period_from_file(train_files[-1])},
    {'control': 'mes_validacion_externa', 'value': VALIDATION_PERIOD},
    {'control': 'validacion_usada_en_fit', 'value': False},
    {'control': 'early_stopping_externo', 'value': False},
    {'control': 'n_estimators_congelado', 'value': FROZEN_N_ESTIMATORS},
    {'control': 'noviembre_diciembre_leidos', 'value': False},
    {'control': 'datos_2023_usados_en_modelizacion', 'value': False},
    {'control': 'filas_train_muestra', 'value': len(train)},
    {'control': 'filas_validacion_externa', 'value': len(validation)},
    {'control': 'modelo_seleccionado', 'value': selected_model},
])
audit_table.to_csv(OUTPUT_DIR / 'auditoria_validacion_externa.csv', index=False, encoding='utf-8-sig')

assert period_from_file(train_files[-1]) == '202209'
assert VALIDATION_PERIOD == '202210'
assert not any(period_from_file(path) >= '202211' for path in train_files + validation_files)
display(selection_result)
display(audit_table)
print(f'Modelo seleccionado tras la validación externa: {selected_model}')

## Lectura del resultado

- Compara la caída o mejora respecto a la media temporal del 06b. Una caída en octubre no invalida automáticamente el modelo, pero debe discutirse como posible deriva temporal.
- Revisa especialmente recall y F1 de `riesgo_vaciado` y `riesgo_saturacion`; la exactitud global está dominada por la clase estable.
- Si XGBoost mantiene la ventaja, queda confirmado como modelo final. La logística permanece como referencia interpretable.
- No ajustes parámetros después de ver octubre. La configuración queda congelada antes de consultar noviembre-diciembre. Los viajes de 2023 quedan únicamente para el análisis descriptivo.